In [1]:
import pandas as pd

df = pd.read_pickle("../data/processed/movies_features.pkl")

threshold = df["vote_count"].quantile(0.75)
df["high_engagement"] = (df["vote_count"] >= threshold).astype(int)

print("Threshold (75th percentile):", threshold)
print(df["high_engagement"].value_counts())

Threshold (75th percentile): 8566.5
high_engagement
0    1289
1     430
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

numeric_features = ["runtime", "budget", "revenue", "popularity", "n_genres", "n_keywords", "overview_length"]
categorical_features = ["original_language", "runtime_category"]

X = df[numeric_features + categorical_features]
y = df["high_engagement"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((1375, 9), (344, 9))

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [4]:
from sklearn.linear_model import LogisticRegression

log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

log_reg_pipeline.fit(X_train, y_train)
print("Trained")

Trained


In [5]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred = log_reg_pipeline.predict(X_test)
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.95      0.89       258
           1       0.77      0.47      0.58        86

    accuracy                           0.83       344
   macro avg       0.81      0.71      0.74       344
weighted avg       0.82      0.83      0.82       344

ROC-AUC: 0.8373895799531278
Confusion matrix:
 [[246  12]
 [ 46  40]]


In [6]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")),
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.90      0.87      0.89       258
           1       0.65      0.72      0.68        86

    accuracy                           0.83       344
   macro avg       0.77      0.79      0.78       344
weighted avg       0.84      0.83      0.83       344

ROC-AUC: 0.8885208220659816
Confusion matrix:
 [[224  34]
 [ 24  62]]


In [7]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=5000)
    )),
])

svm_pipeline.fit(X_train, y_train)

y_pred_svm = svm_pipeline.predict(X_test)
y_proba_svm = svm_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_svm))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_svm))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_svm))

              precision    recall  f1-score   support

           0       0.84      0.95      0.89       258
           1       0.76      0.44      0.56        86

    accuracy                           0.83       344
   macro avg       0.80      0.70      0.73       344
weighted avg       0.82      0.83      0.81       344

ROC-AUC: 0.8418514512349018
Confusion matrix:
 [[246  12]
 [ 48  38]]


In [8]:
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    rf_pipeline, X, y, cv=skf,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)

for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    print(metric, "mean:", round(scores[metric].mean(), 3), "std:", round(scores[metric].std(), 3))

test_accuracy mean: 0.828 std: 0.007
test_precision mean: 0.644 std: 0.016
test_recall mean: 0.698 std: 0.035
test_f1 mean: 0.669 std: 0.016
test_roc_auc mean: 0.882 std: 0.012


In [11]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    rf_pipeline, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)   # train split only

print("Best params:", grid_search.best_params_)
print("Best F1 (cross-val):", grid_search.best_score_)

Best params: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 4, 'classifier__n_estimators': 300}
Best F1 (cross-val): 0.6950692640692642


In [12]:
best_rf_pipeline = grid_search.best_estimator_

y_pred_tuned = best_rf_pipeline.predict(X_test)
y_proba_tuned = best_rf_pipeline.predict_proba(X_test)[:, 1]

print("--- Default Random Forest (before) ---")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

print("\n--- Tuned Random Forest (after) ---")
print(classification_report(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_tuned))

--- Default Random Forest (before) ---
              precision    recall  f1-score   support

           0       0.90      0.87      0.89       258
           1       0.65      0.72      0.68        86

    accuracy                           0.83       344
   macro avg       0.77      0.79      0.78       344
weighted avg       0.84      0.83      0.83       344

ROC-AUC: 0.8885208220659816

--- Tuned Random Forest (after) ---
              precision    recall  f1-score   support

           0       0.93      0.82      0.87       258
           1       0.60      0.83      0.70        86

    accuracy                           0.82       344
   macro avg       0.77      0.82      0.78       344
weighted avg       0.85      0.82      0.83       344

ROC-AUC: 0.8888588426176312


In [13]:
import joblib

joblib.dump(best_rf_pipeline, "../models/classifier_rf.pkl")
print("saved")

saved
